In [ ]:
!pip install category_encoders

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Types ofEncoder

1. Label encoder--Each category gets a unique number

Advantages:

Very simple

Uses only one column

Disadvantages:

It introduces an artificial order.

The model may think:

Red > Green > Blue

In [ ]:
df=pd.DataFrame({'color':['red','blue','green']})
df

,color
0,red
1,blue
2,green


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["color"] = le.fit_transform(df["color"])
df

,color
0,2
1,0
2,1


2. OneHotEncoding--Creates one new column for every category

Advantages

No fake ordering.

Disadvantages

Many categories create many columns.

In [ ]:
df = pd.get_dummies(df, columns=["color"])
df

,color_0,color_1,color_2
0,False,False,True
1,True,False,False
2,False,True,False


3. Ordinal Encoding--Used when categories have a natural order

Advantages:

Keeps the real order

Disadvantages:

Wrong if categories aren't actually ordered.

In [ ]:
df=pd.DataFrame({'size':['small','medium','large']})
df

,size
0,small
1,medium
2,large


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(categories=[["small","medium","large"]])

df["size"] = encoder.fit_transform(df[["size"]])
df

,size
0,0.0
1,1.0
2,2.0


4. BinaryEncoder--same as OneHotEncoding .BinaryEncoder uses binary representation and uses fewer column than OneHotEncoding

In [ ]:
import pandas as pd
from category_encoders import BinaryEncoder

df = pd.DataFrame({
    "Fruit": ["Apple", "Banana", "Orange", "Mango", "Banana", "Apple"]
})

encoder = BinaryEncoder(cols=["Fruit"])

df_encoded = encoder.fit_transform(df)

print(df_encoded)

   Fruit_0  Fruit_1  Fruit_2
0        0        0        1
1        0        1        0
2        0        1        1
3        1        0        0
4        0        1        0
5        0        0        1


5. Frequency encoding--Frequency encoding replaces each category with how often it appears in the dataset. Unlike count encoding, it usually uses the proportion (percentage) instead of the raw count.

In [ ]:
df = pd.DataFrame({
    "Animal": ["Dog", "Cat", "Dog", "Cow", "Dog", "Cat"]
})

# Frequency Encoding
freq = df["Animal"].value_counts(normalize=True)

df["Animal"] = df["Animal"].map(freq)

print(df)
'''Frequency = Count ÷ Total Rows

Animal	Frequency
Dog	     3/6 = 0.50
Cat	     2/6 = 0.33
Cow	     1/6 = 0.1667
'''

     Animal
0  0.500000
1  0.333333
2  0.500000
3  0.166667
4  0.500000
5  0.333333


6. Count Encoding--Count encoding replaces each category with the number of times it appears in the dataset

In [ ]:
df = pd.DataFrame({
    "City": ["London", "Paris", "London", "Tokyo", "London", "Paris"]
})

count_map = df["City"].value_counts()

df["City"] = df["City"].map(count_map)

print(df)

   City
0     3
1     2
2     3
3     1
4     3
5     2


7. Target/Mean Encoding--Target encoding replaces each category with the average value of the target variable for that category.

In [ ]:
'''

Suppose you're predicting whether a customer buys a product.

Original dataframe:

City	     Bought (Target)
London	         1
London	         0
London	         1
Paris	           0
Paris	           0
Tokyo	           1

Step 1: Calculate the mean target for each city
formula=sum of target values for that category /repetation of that catego
For London
(1 + 0 + 1) / 3 = 0.67

For Paris
(0 + 0) / 2 = 0

For Tokyo
1 / 1 = 1

Step 2: Replace the categories
City	      Encoded
London	    0.67
London	    0.67
London	    0.67
Paris	      0.00
Paris	      0.00
Tokyo	      1.00
'''

In [ ]:
df = pd.DataFrame({
    "City": ["London", "London", "London", "Paris", "Paris", "Tokyo"],
    "Bought": [1, 0, 1, 0, 0, 1]
})

target_mean = df.groupby("City")["Bought"].mean()

df["City"] = df["City"].map(target_mean)

print(df)

       City  Bought
0  0.666667       1
1  0.666667       0
2  0.666667       1
3  0.000000       0
4  0.000000       0
5  1.000000       1


# **Train-test split**

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Load dataset
iris = load_iris()

X = iris.data       # Features
y = iris.target     # Target

# 2. Split data into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Create the model
model = DecisionTreeClassifier()

# 4. Train the model
model.fit(X_train, y_train)

# 5. Make predictions
y_pred = model.predict(X_test)

# 6. Check accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

# Train-test validation split


* train=training data
* validation=data used to choose which model is the best or tune the data
* test =testing data

In [ ]:
Training data
      ↓
Train Model A
      ↓
Validation data → 85% accuracy


Training data
      ↓
Train Model B
      ↓
Validation data → 91% accuracy

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
iris = load_iris()
X = iris.data
y = iris.target

# ------------------------------------------------
# 1. Split data: 70% Train, 15% Validation, 15% Test
# ------------------------------------------------
#first split (train vs(val+test))
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)
#second split (val vs test)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

# ------------------------------------------------
# 2. Create two models
# ------------------------------------------------

decision_tree = DecisionTreeClassifier(random_state=42)

random_forest = RandomForestClassifier(random_state=42)

# ------------------------------------------------
# 3. Train both models using SAME training data
# ------------------------------------------------

decision_tree.fit(X_train, y_train)
random_forest.fit(X_train, y_train)

# ------------------------------------------------
# 4. Check both models on SAME validation data
# ------------------------------------------------

dt_pred = decision_tree.predict(X_val)
rf_pred = random_forest.predict(X_val)

dt_accuracy = accuracy_score(y_val, dt_pred)
rf_accuracy = accuracy_score(y_val, rf_pred)
print('*'*40)
print("Decision Tree Validation Accuracy:", dt_accuracy)
print("Random Forest Validation Accuracy:", rf_accuracy)
print('*'*40)

# ------------------------------------------------
# 5. Choose the better model
# ------------------------------------------------

if dt_accuracy > rf_accuracy:
    best_model = decision_tree
    print("Best Model: Decision Tree")
else:
    best_model = random_forest
    print("Best Model: Random Forest")
print('*'*40)

# ------------------------------------------------
# 6. Test the BEST model on test data
# ------------------------------------------------

test_prediction = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, test_prediction)

print("Final Test Accuracy:", test_accuracy)

****************************************
Decision Tree Validation Accuracy: 1.0
Random Forest Validation Accuracy: 1.0
****************************************
Best Model: Random Forest
****************************************
Final Test Accuracy: 1.0
